In [42]:
from dotenv import load_dotenv
import os
import textwrap
from langchain_neo4j import Neo4jGraph, Neo4jVector
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.chains import RetrievalQAWithSourcesChain

#warning controls
import warnings
warnings.filterwarnings("ignore")

In [2]:
load_dotenv('.env', override=True)
NEO4J_URI = os.getenv('NEO4J_URI')
NEO4J_USERNAME = os.getenv('NEO4J_USERNAME')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD')
NEO4J_DATABASE = os.getenv('NEO4J_DATABASE') or 'neo4j'
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
# Note the code below is unique to this course environment, and not a 
# standard part of Neo4j's integration with OpenAI. Remove if running 
# in your own environment.
OPENAI_ENDPOINT = os.getenv('OPENAI_BASE_URL') + '/embeddings'

VECTOR_INDEX_NAME = 'form_10k_chunks'
VECTOR_NODE_LABEL = "Chunk"
VECTOR_SOURCE_PROPERTY = "text"
VECTOR_EMBEDDING_PROPERTY = "textEmbedding"

In [3]:
kg = Neo4jGraph(
    url=NEO4J_URI, username=NEO4J_USERNAME, password=NEO4J_PASSWORD, database=NEO4J_DATABASE
)

In [12]:
kg.query("""
         CREATE VECTOR INDEX `form_10k_chunks` IF NOT EXISTS
         FOR (c:Chunk) ON (c.textEmbedding)
         OPTIONS { indexConfig: {
             `vector.dimensions`: 1536,
             `vector.similarity_function`: 'cosine'
         }}
""")

[]

In [13]:
kg.query("show indexes")

[{'id': 6,
  'name': 'form_10k_chunks',
  'state': 'ONLINE',
  'populationPercent': 100.0,
  'type': 'VECTOR',
  'entityType': 'NODE',
  'labelsOrTypes': ['Chunk'],
  'properties': ['textEmbedding'],
  'indexProvider': 'vector-2026.06',
  'owningConstraint': None,
  'lastRead': None,
  'readCount': 0},
 {'id': 1,
  'name': 'index_343aff4e',
  'state': 'ONLINE',
  'populationPercent': 100.0,
  'type': 'LOOKUP',
  'entityType': 'NODE',
  'labelsOrTypes': None,
  'properties': None,
  'indexProvider': 'token-lookup-1.0',
  'owningConstraint': None,
  'lastRead': neo4j.time.DateTime(2026, 7, 31, 21, 47, 39, 422000000, tzinfo=<UTC>),
  'readCount': 184},
 {'id': 2,
  'name': 'index_f7700477',
  'state': 'ONLINE',
  'populationPercent': 100.0,
  'type': 'LOOKUP',
  'entityType': 'RELATIONSHIP',
  'labelsOrTypes': None,
  'properties': None,
  'indexProvider': 'token-lookup-1.0',
  'owningConstraint': None,
  'lastRead': neo4j.time.DateTime(2024, 1, 18, 11, 46, 2, 950000000, tzinfo=<UTC>),
  

In [19]:
#Calculate embedding vectors for chunks and populate index
# This query calculates the embedding vector and stores it as a property called textEmbedding on each Chunk node.

kg.query("""
    match (c:Chunk) where c.textEmbedding is null
    with c, genai.vector.encode(c.text, "OpenAI",
    {
        token: $openAIKey,
        endpoint: $openAIEndpoint
    }) as vector
    call db.create.setNodeVectorProperty(c, "textEmbedding", vector)
""", params={"openAIKey": OPENAI_API_KEY, "openAIEndpoint": OPENAI_ENDPOINT})

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. genai.vector.encode is deprecated. It is replaced by ai.text.embed.', position=<SummaryInputPosition line=3, column=13, offset=63>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 63, 'line': 3, 'column': 13}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n    match (c:Chunk) where c.textEmbedding is null\n    with c, genai.vector.encode(c.text, "OpenAI",\n    {\n        token: $openAIKey,\n        endpoint: $openAIEndpoint\n    }) as vector\n    call db.create.setNodeVectorProperty(c, "textEmbedding", vector)\n'


[]

In [21]:
kg.refresh_schema()
print(kg.schema)

Node properties:
Movie {title: STRING, taglineEmbedding: LIST, tagline: STRING, released: INTEGER}
Person {born: INTEGER, name: STRING, lastSeen: INTEGER}
Chunk {f10kItems: STRING, source: STRING, textEmbedding: LIST, cik: STRING, cusip6: STRING, chunkId: STRING, text: STRING, chunkSeqId: INTEGER, formId: STRING}
Relationship properties:
ACTED_IN {roles: LIST}
REVIEWED {summary: STRING, rating: INTEGER}
The relationships:
(:Person)-[:ACTED_IN]->(:Movie)
(:Person)-[:DIRECTED]->(:Movie)
(:Person)-[:PRODUCED]->(:Movie)
(:Person)-[:WROTE]->(:Movie)
(:Person)-[:FOLLOWS]->(:Person)
(:Person)-[:REVIEWED]->(:Movie)
(:Person)-[:KNOWS]->(:Person)


Use similarity search to find relevant chunks

In [30]:
def neo4j_vector_search(question):
    """Search for similar nodes using the vector index"""
    vector_search_query="""with genai.vector.encode($question, "OpenAI", {token: $openAIKey, endpoint: $openAIEndpoint}) as questionEmbedding
    call db.index.vector.queryNodes($indexName, $top_k, questionEmbedding)  yield node, score
    return score, node.text as text
    """
    
    similar = kg.query(vector_search_query, params={"question": question,
        "openAIKey": OPENAI_API_KEY,
        "openAIEndpoint": OPENAI_ENDPOINT,
        "indexName": VECTOR_INDEX_NAME,
        "top_k": 10
    })
    return similar

In [33]:
search_results = neo4j_vector_search('In a single sentence, tell me about Netapp.')

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. genai.vector.encode is deprecated. It is replaced by ai.text.embed.', position=<SummaryInputPosition line=1, column=6, offset=5>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 5, 'line': 1, 'column': 6}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'with genai.vector.encode($question, "OpenAI", {token: $openAIKey, endpoint: $openAIEndpoint}) as questionEmbedding\n    call db.index.vector.queryNodes($indexName, $top_k, questionEmbedding)  yield node, score\n    return score, node.text as text\n    '
Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_descr

In [34]:
search_results[0]

{'score': 0.9357028007507324,
 'text': '>Item 1.  \nBusiness\n\n\nOverview\n\n\nNetApp, Inc. (NetApp, we, us or the Company) is a global cloud-led, data-centric software company. We were incorporated in 1992 and are headquartered in San Jose, California. Building on more than three decades of innovation, we give customers the freedom to manage applications and data across hybrid multicloud environments. Our portfolio of cloud services, and storage infrastructure, powered by intelligent data management software, enables applications to run faster, more reliably, and more securely, all at a lower cost.\n\n\nOur opportunity is defined by the durable megatrends of data-driven digital and cloud transformations. NetApp helps organizations meet the complexities created by rapid data and cloud growth, multi-cloud management, and the adoption of next-generation technologies, such as AI, Kubernetes, and modern databases. Our modern approach to hybrid, multicloud infrastructure and data managemen

In [39]:
neo4j_vector_store = Neo4jVector.from_existing_graph(
    embedding=OpenAIEmbeddings(),
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    index_name=VECTOR_INDEX_NAME,
    node_label=VECTOR_NODE_LABEL,
    text_node_properties=[VECTOR_SOURCE_PROPERTY],
    embedding_node_property=VECTOR_EMBEDDING_PROPERTY,
)

In [40]:
retriever = neo4j_vector_store.as_retriever()

In [46]:
# chain = RetrievalQAWithSourcesChain.from_chain_type(
#     ChatOpenAI(temperature=0, model_name="gpt-4o-mini", openai_api_key=OPENAI_API_KEY, openai_api_base=OPENAI_ENDPOINT),
#     retriever=retriever,
#     chain_type="stuff"
# )
chain = RetrievalQAWithSourcesChain.from_chain_type(
    ChatOpenAI(temperature=0), 
    chain_type="stuff", 
    retriever=retriever
)

In [47]:
def prettyChain(question):
    """"Pretty print the chains response to question"""
    response = chain({"question": question}, return_only_outputs=True)
    print(response["answer"])
    print(textwrap.fill(response['answer'], 60))

In [48]:
question = "What is Netapp's primary business?"
prettyChain(question)

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL db.index.vector.queryNodes($vector_index_name, $top_k * $effective_search_ratio, $query_vector) YIELD node, score WITH node, score LIMIT $top_k RETURN reduce(str='', k IN ['text'] | str + '\\n' + k + ': ' + coalesce(node[k], '')) AS text, node {.*, `textEmbedding`: Null, `text`: Null} AS metadata, score"


NetApp's primary business is enterprise storage and data management, cloud storage, and cloud operations.

NetApp's primary business is enterprise storage and data
management, cloud storage, and cloud operations.
